In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

print("X_train_t:", X_train_t.shape, X_train_t.dtype)
print("y_train_t:", y_train_t.shape, y_train_t.dtype)

In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t, y_test_t)


In [ ]:
# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

In [ ]:
# 4. Print shape of one batch
X_batch, y_batch = next(iter(train_loader))
print("\nOne batch shapes:")
print("X_batch:", X_batch.shape)
print("y_batch:", y_batch.shape)

In [ ]:
# 5. Display sample images
plt.figure(figsize=(10, 4))
for i in range(6):
    img = X_batch[i].permute(1, 2, 0).numpy()
    age = y_batch[i].item()
    plt.subplot(2, 3, i+1)
    plt.imshow(img)
    plt.title(f"Age: {age:.1f}")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1) Model class with 4 linear layers
class AgeRegressor4Layer(nn.Module):
    def __init__(self, input_dim=3*36*36, h1=256, h2=128, h3=64, output_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, h1),      # 1
            nn.ReLU(),
            nn.Linear(h1, h2),             # 2
            nn.ReLU(),
            nn.Linear(h2, h3),             # 3
            nn.ReLU(),
            nn.Linear(h3, output_dim)      # 4
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    n = 0

    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(Xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * Xb.size(0)
        n += Xb.size(0)

    return running_loss / n


In [ ]:
# Task 3: Write your validation loop here:
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    n = 0

    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        preds = model(Xb)
        loss = criterion(preds, yb)

        running_loss += loss.item() * Xb.size(0)
        n += Xb.size(0)

    return running_loss / n


In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeRegressor4Layer().to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print("Device:", device)
print(model)



In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20
train_losses, val_losses = [], []

for epoch in range(1, num_epochs + 1):
    tr_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va_loss = validate_one_epoch(model, test_loader, criterion, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)

    print(f"Epoch {epoch:02d}/{num_epochs} | Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here:
model.eval()

Xb, yb = next(iter(test_loader))
Xb = Xb.to(device)
yb = yb.to(device)

with torch.no_grad():
    preds = model(Xb)

n_show = 6
plt.figure(figsize=(10, 4))

for i in range(n_show):
    img = Xb[i].permute(1, 2, 0).detach().cpu().numpy()
    actual = yb[i].item()
    pred = preds[i].item()

    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(f"Pred: {pred:.1f} | Actual: {actual:.1f}")
    plt.axis("off")

plt.tight_layout()
plt.show()
